# IWDD 2027 test
Runtime: GPU (T4 or better). Set `ROOT` to this repository's folder on your Drive, `VIDEOS` to the test videos and `RESULTS`
to the folder for the result files (created if missing). The notebook installs the pinned packages of `requirements.txt` (torch: Colab's
own), then runs `python test.py --videos ... --results ...` from `ROOT`, as the official sample does, and prints every
result file. The model files ship in `submission/model/` (SHA256 checked at load); nothing is downloaded.

In [ ]:
ROOT = '/content/drive/MyDrive/iwdd2027'   # this repository (test.py at its root)
VIDEOS = '/content/drive/MyDrive/videos'   # the test videos
RESULTS = '/content/drive/MyDrive/results' # the result TXTs (created if missing; nothing is deleted)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
assert os.path.isfile(f'{ROOT}/test.py'), f'ROOT = {ROOT!r}: set it to the iwdd2027 folder (test.py at its root)'
# Pinned: PyAV, the OpenCV build used in training (Colab's preinstalled OpenCV builds removed first), einops.
!pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless
!pip install -q -r "{ROOT}/requirements.txt"

In [ ]:
import os, subprocess, time
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
assert GPU, 'no GPU: set Runtime > Change runtime type > GPU (without one, test.py writes the constant 13 s)'
os.makedirs(RESULTS, exist_ok=True)  # test.py writes one TXT per video (a TXT of the same name is overwritten)
%cd "{ROOT}"
t0 = time.time()
!python test.py --videos "{VIDEOS}/" --results "{RESULTS}/" 2>&1 | tee "{ROOT}/test_results.log"
WALL = time.time() - t0

In [ ]:
# Summary: GPU, versions, policy and failure count, wall time, every result file.
LOG = open(f'{ROOT}/test_results.log').read().splitlines()
print('GPU:', GPU)
print(next(l for l in LOG if l.startswith('torch ')).strip())
print(next(l for l in LOG if l.startswith('policy ')).strip())
print(f'wall time of test.py (incl. model load): {WALL:.0f} s')
for n in sorted(os.listdir(RESULTS)):
    print(n, repr(open(os.path.join(RESULTS, n)).read()))